In [ ]:
# 대회 환경 맞추기
!pip uninstall -y torch transformers -q

!pip install torch
!pip install transformers==4.57.3
!pip install tokenizers==0.22.1
!pip install accelerate==1.10.1
!pip install datasets==4.4.1
!pip install compressed-tensors==0.13.0

# llmcompressor 설치 시도
!pip install llmcompressor

print("설치 완료 - 런타임 재시작 필요!")


  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (915.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
compressed-tensors 0.13.0 requires transformers, which is not installed.
peft 0.18.1 requires transformers, which is not installed.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, which is not installed.
torchvision 0.24.0+cu128 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
torchaudio 2.9.0+cu128 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which is incompatible.
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
  Using cached llmcompressor-0.9.0

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier


In [ ]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 512 # 캘리브레이션 데이터 수 증가 256-> 512
MAX_SEQUENCE_LENGTH = 1024 # 512 -> 1024

# Quantization
SCHEME = "W4A16" # GPTQ-int4
TARGETS = ["Linear"]
#IGNORE  = ["embed_tokens", "lm_head"] -> ignore하는 대상 헷갈리지 않게 하기 위해


In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16, #bfloat16-> float16
)

print("[INFO] 모델/토크나이저 로드 완료")


[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


In [ ]:
# (추가) 모든 층 down_proj + 마지막 4개 층 보호
n_layers = model.config.num_hidden_layers
ignore_list = ["model.embed_tokens", "model.norm", "lm_head"]

for i in range(n_layers):
    # down_proj
    ignore_list.append(f"model.layers.{i}.mlp.down_proj")

    # 26.27.28.29(에러 많은 레이어)
    if i >= n_layers - 2: # 4->2
        ignore_list.append(f"model.layers.{i}.self_attn.q_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.k_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.v_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.o_proj")
        ignore_list.append(f"model.layers.{i}.mlp.gate_proj")
        ignore_list.append(f"model.layers.{i}.mlp.up_proj")

print(f"[DEBUG] 제외 대상(IGNORE) 설정 완료: 총 {len(ignore_list)}개 모듈 제외됨")
print(f"[INFO] Layer {n_layers-2} ~ {n_layers-1} (총 2개 층)은 완전히 보호됩니다.")


[DEBUG] 제외 대상(IGNORE) 설정 완료: 총 45개 모듈 제외됨
[INFO] Layer 28 ~ 29 (총 2개 층)은 완전히 보호됩니다.


In [ ]:
print("[INFO] 데이터 전처리 중...")

from itertools import chain
from datasets import Dataset

# 원본에서 넉넉히 뽑아서(=NUM_CALIBRATION_SAMPLES * 5) 셔플 후 사용
raw_ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
raw_ds = raw_ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES * 5))

def get_packed_dataset(dataset, tokenizer, seq_len=2048, num_samples=512):
    formatted_texts = []
    for entry in dataset:
        try:
            text = tokenizer.apply_chat_template(
                entry["conversations"],
                tokenize=False,
                add_generation_prompt=True
            )
            formatted_texts.append(text)
        except Exception:
            continue

    encodings = tokenizer(formatted_texts, add_special_tokens=False)
    all_input_ids = list(chain(*encodings["input_ids"]))

    packed_dataset = []
    for i in range(0, len(all_input_ids), seq_len):
        chunk_ids = all_input_ids[i:i + seq_len]
        if len(chunk_ids) == seq_len:
            packed_dataset.append({
                "input_ids": chunk_ids,
                "attention_mask": [1] * seq_len,
            })
        if len(packed_dataset) >= num_samples:
            break

    return packed_dataset

ds_list = get_packed_dataset(
    raw_ds,
    tokenizer,
    seq_len=MAX_SEQUENCE_LENGTH,              # 네가 쓰는 max len
    num_samples=NUM_CALIBRATION_SAMPLES       # 캘리브레이션 샘플 수
)

ds = Dataset.from_list(ds_list)

print("[INFO] 데이터 전처리 완료")
print("[INFO] packed samples:", len(ds), "| max_len:", MAX_SEQUENCE_LENGTH)


[INFO] 데이터 전처리 중...
[INFO] 데이터 전처리 완료
[INFO] packed samples: 512 | max_len: 1024


In [ ]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_list,
        dampening_frac=0.2, # dampening_frac = 0.1 -> 0.2
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")


[INFO] GPTQ 시작 (scheme=W4A16, samples=512, max_len=1024)...
2026-02-16T12:41:03.703185+0000 | reset | INFO - Compression lifecycle reset
2026-02-16T12:41:03.705799+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-16T12:41:03.773234+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-16T12:41:03.773959+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 512/512 [00:05<00:00, 97.25it/s] 

2026-02-16T12:41:10.215526+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 512 samples


2026-02-16T12:41:11.717440+0000 | compress | METRIC - time 1.50s
2026-02-16T12:41:11.718224+0000 | compress | METRIC - error 4.07
2026-02-16T12:41:11.719188+0000 | compress | METRIC - GPU 0 | usage: 6.24% | total memory: 24 GB
2026-02-16T12:41:11.719759+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-16T12:41:11.720828+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.k_proj using 512 samples
2026-02-16T12:41:12.843379+0000 | compress | METRIC - time 1.12s
2026-02-16T12:41:12.844215+0000 | compress | METRIC - error 1.19
2026-02-16T12:41:12.845092+0000 | compress | METRIC - GPU 0 | usage: 6.24% | total memory: 24 GB
2026-02-16T12:41:12.845589+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-16T12:41:12.846633+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.v_proj using 512 samples
2026-02-16T12:41:13.960381+0000 | compress | METRIC - time 1.11s
2026-02-16T12:41:13.961451+0000 | compress | METRIC - error

(2/31): Calibrating: 100%|██████████| 512/512 [00:04<00:00, 102.82it/s]

...

(28/31): Calibrating: 100%|██████████| 512/512 [00:05<00:00, 99.54it/s]

2026-02-16T12:47:38.254097+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.q_proj using 512 samples


2026-02-16T12:47:39.405505+0000 | compress | METRIC - time 1.15s
2026-02-16T12:47:39.406803+0000 | compress | METRIC - error 7390.18
2026-02-16T12:47:39.407725+0000 | compress | METRIC - GPU 0 | usage: 6.24% | total memory: 24 GB
2026-02-16T12:47:39.408521+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-16T12:47:39.409895+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.k_proj using 512 samples
2026-02-16T12:47:40.544766+0000 | compress | METRIC - time 1.13s
2026-02-16T12:47:40.546093+0000 | compress | METRIC - error 1937.69
2026-02-16T12:47:40.546877+0000 | compress | METRIC - GPU 0 | usage: 6.24% | total memory: 24 GB
2026-02-16T12:47:40.547322+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-16T12:47:40.548525+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.v_proj using 512 samples
2026-02-16T12:47:41.679583+0000 | compress | METRIC - time 1.13s
2026-02-16T12:47:41.680895+0000 | compress | METRIC

(31/31): Propagating: 100%|██████████| 512/512 [00:00<00:00, 726.05it/s]

2026-02-16T12:47:57.085685+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-16T12:47:57.131830+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ 완료


In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")


2026-02-16T12:48:32.408164+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 168it [00:01, 117.04it/s]


[INFO] 모델 저장 완료: ./model


In [ ]:
zip_name = "gptq2"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")


[INFO] gptq2.zip 생성 중...
[INFO] 생성 완료: gptq2.zip


In [ ]:
from google.colab import files
files.download("gptq2.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#정확도 평가

In [ ]:
!pip -q install lm-eval[vllm]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 112.9 MB/s eta 0:00:00
   ━━━

In [ ]:
!python -m lm_eval --model vllm \
  --model_args pretrained=/content/model,gpu_memory_utilization=0.85,max_model_len=2048 \
  --tasks gsm8k \
  --limit 512 \
  --apply_chat_template \
  --batch_size auto


2026-02-16:12:53:41 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-02-16:12:53:41 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-16:12:53:48 INFO     [_cli.run:376] Selected Tasks: ['gsm8k']
2026-02-16:12:53:48 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-16:12:53:48 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': '/content/model', 'gpu_memory_utilization': 0.85, 'max_model_len': 2048}
2026-02-16 12:53:55.294885: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771246435.318881    5941 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for p

#속도 평가

In [ ]:
!pip -q install vllm==0.14.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 150.7 MB/s eta 0:00:00


In [ ]:
!git clone -q https://github.com/vllm-project/vllm.git


In [ ]:
!vllm bench throughput \
  --model /content/model \
  --dataset-name random \
  --num-prompts 100 \
  --random-input-len 100 \
  --random-output-len 200 \
  --random-range-ratio 0.0 \
  --seed 42 \
  --gpu-memory-utilization 0.85


2026-02-16 12:59:02.974058: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771246742.999284    7936 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771246743.006662    7936 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771246743.024360    7936 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771246743.024404    7936 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771246743.024407    7936 computation_placer.cc:177] computation placer alr

<방법 1>
1. ignore랑 ignore_list가 같이 있으면 ignore의 대상이 헷갈릴 수 있기 때문에 ignore지우기
2. 캘리브레이션 데이터 로드 부분에서 packed를 써서 긴 샘플이 들어와도 activation 분포가 일정해짐.
3. 마지막 4층 보호를 2층 보호로 줄이기(속도 측면 고려)
4. dampening_frac=0.2로 지정하기

속도: 4938.47 output tokens/s

용량: 1.35GB
-> dampening_frac=0.1로 했더니 1.39GB




성능:
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.2598|±  |0.0194|
|     |       |strict-match    |     5|exact_match|↑  |0.0645|±  |0.0109|

<방법 2>
1. packed off
2. dampening_frac=0.2,       
3. ignore_list = ["model.embed_tokens", "model.norm", "lm_head"]
4. 마지막 4개 층 보호

속도: 4711.89 output tokens/s

용량: 1.3GB

성능:

|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.2773|±  |0.0198|
|     |       |strict-match    |     5|exact_match|↑  |0.0469|±  |0.0094|

<방법 3>
1. torch_dtype=bfloat16에서 float16으로 바꾸기

속도: 4727.77 output tokens/s

용량: 1.27G



성능:
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.3105|±  |0.0205|
|     |       |strict-match    |     5|exact_match|↑  |0.0527|±  |0.0099|

<방법 4>
1. torch_dtype=bfloat16에서 float16으로 바꾸기
2.
n_layers = model.config.num_hidden_layers
- embed/norm 보호 제거해도 됨 (Linear 타겟만 양자화라 영향 없음)

ignore_list = ["lm_head"]


- down_proj는 마지막 8개 레이어만 보호

for i in range(n_layers):
    if i >= n_layers - 8:
        ignore_list.append(f"model.layers.{i}.mlp.down_proj")

- 마지막 2개 레이어는 여전히 full 보호



속도: 4912.22 output tokens/s

용량: 1.17G


성능:
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.2402|±  |0.0189|
|     |       |strict-match    |     5|exact_match|↑  |0.0195|±  |0.0061|